In [1]:
import requests #api fetch
import pandas as pd #data handling
import numpy as np #num operations
from sklearn.model_selection import train_test_split #splitting data
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor #regression models
from sklearn.metrics import mean_squared_error #error finding
from datetime import datetime, timedelta 
import pytz #timezones

In [2]:
API_KEY='7183784f17e8101a6f4779788eada597'
BASE_URL= 'https://api.openweathermap.org/data/2.5/'

Fetching todays weather data

In [3]:
def get_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric" #request format
    reply = requests.get(url) #sening to api
    data= reply.json() #converting to json file
    precip = 0
    if 'rain' in data:
        precip = data['rain'].get('1h', 0) #checking if rain data is present
    return {
        'city': data['name'],
        'feels_like': round(data['main']['feels_like']),
        'temp_mean': round(data['main']['temp']),
        'temp_min': round(data['main']['temp_min']),
        'temp_max': round(data['main']['temp_max']),
        'humidity': round(data['main']['humidity']),
        'wind_dir': data['wind']['deg'],
        'pressure': data['main']['pressure'],
        'wind_gusts': round(data['wind']['speed']),
        'precip_sum': round(precip, 2),
        'info': data['weather'][0]['description'],
        'country': data['sys']['country'],
    }

Reading Old Data

In [4]:
def read_olddata(file):
    df= pd.read_csv(file) 
    df=df.dropna()
    df=df.drop_duplicates()
    return df

Prepping data

In [5]:
def dataprep(data):
    
    x= data[['temp_mean', 'temp_max', 'temp_min', 'wind_dir', 'humidity', 'pressure', 'wind_gusts','precip_sum']]
    y= data['target']
    return x, y

Training Rain model

In [ ]:
def train_rain_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) #splitting data
    model= RandomForestClassifier(n_estimators=100, random_state=42, class_weight={0: 1, 1: 3}) #random forest model
    model.fit(x_train, y_train) #training model
    
    y_pred = model.predict(x_test) #making prediction on test data
    
    #mean square error for rain pred
    print(mean_squared_error(y_test, y_pred))
    return model

regression data 

In [7]:
def prep_regressiondata(data, features):
    x, y =[], []
    for i in range(len(data)-1):
        x.append(data[features].iloc[i])
        y.append(data[features].iloc[i+1])
        
    x=np.array(x).reshape(-1,1)
    y=np.array(y)
    
    return x,y

Training regression model

In [8]:
def train_regressionmodel(x,y):
    model = RandomForestRegressor(n_estimators=100, random_state=42) #regression model
    model.fit(x, y) #training model
    return model


Predictions

In [9]:
def predict(model, curr_val):
    predictions = [curr_val]
    
    for i in range(5):
        nextval= model.predict(np.array([[predictions[-1]]]))
        
        predictions.append(nextval[0])
    return predictions[1:]

In [10]:
def weather_forecast():
    print("Welcome to the Weather Forecasting App")
    city = input('Enter your city name: ')
    print(city)
    todayweather = get_weather(city)
    
    old_data = read_olddata('finalopenmeteo.csv')
    
    x, y = dataprep(old_data)
    
    rain = train_rain_model(x, y)
    
    curr_data= {
        'temp_mean': todayweather['temp_mean'],
        'temp_max': todayweather['temp_max'],
        'temp_min': todayweather['temp_min'],
        'wind_dir': todayweather['wind_dir'],
        'humidity': todayweather['humidity'],
        'pressure': todayweather['pressure'],
        'wind_gusts': todayweather['wind_gusts'],
        'precip_sum': todayweather['precip_sum'],
    }
    
    current_df = pd.DataFrame([curr_data])
    rain_pred = rain.predict(current_df)[0]
    
    #temp and hum
    x_tempmax, y_tempmax = prep_regressiondata(old_data, ['temp_max'])
    x_hum, y_hum = prep_regressiondata(old_data, ['humidity'])
    
    tempmaxmodel = train_regressionmodel(x_tempmax, y_tempmax)
    hummodel= train_regressionmodel(x_hum, y_hum)

    tempmax_pred = predict(tempmaxmodel, todayweather['temp_max'])
    hum_pred = predict(hummodel, todayweather['humidity'])

    timezone= pytz.timezone('Asia/Kolkata')
    time= datetime.now(timezone)
    
    futuredates = [(time + timedelta(days=i+1)).strftime('%d-%b') for i in range(5)]
    
    print(f"City: {city}", {todayweather['country']})
    print(f"Current Temperature: {todayweather['temp_mean']}" )
    print(f"Feels Like: {todayweather['feels_like']}" )
    print(f"Minimum Temperature: {todayweather['temp_min']}°C" )
    print(f"Maximum Temperature: {todayweather['temp_max']}°C" )
    print(f"Humidity: {todayweather['humidity']}%" )
    
    print("Weather Prediction")
    print(f"weather: {todayweather['info']}" )
    print(f"Rain: {'YES' if rain_pred else 'NO'}" )
      
    print("\n5-Day Maximum Temperature Forecast at this time:")
    for date, tempmax in zip(futuredates, tempmax_pred):
        print(f"Date {date}: {round(tempmax,1)}°C")

    print("\n5-Day Humidity Forecast at this time:")
    for date, humidity in zip(futuredates, hum_pred):  
        print(f"Date {date}: {round(humidity,1)}%")


    

In [11]:
weather_forecast()

Welcome to the Weather Forecasting App
patiala
check
0.0


c:\Users\Shivanshu Sharma\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Shivanshu Sharma\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


City: patiala {'IN'}
Current Temperature: 26
Feels Like: 26
Minimum Temperature: 26°C
Maximum Temperature: 26°C
Humidity: 85%
Weather Prediction
weather: broken clouds
Rain: NO

5-Day Maximum Temperature Forecast at this time:
Date 19-Sep: 26.1°C
Date 20-Sep: 26.3°C
Date 21-Sep: 26.6°C
Date 22-Sep: 27.1°C
Date 23-Sep: 27.5°C

5-Day Humidity Forecast at this time:
Date 19-Sep: 84.2%
Date 20-Sep: 83.3%
Date 21-Sep: 82.3%
Date 22-Sep: 81.6%
Date 23-Sep: 81.6%
